# R00 · Reproduce everything

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-account>/nids-partition-distortion-ae-dae/blob/main/notebooks/reproduction/R00_Run_All_Reproductions.ipynb)

Runs R01–R07 in order in this runtime: every manuscript check, the figures, the claims map and the test suite. Expected: four claim logs with 0 failures, five figures, and 82 tests passed / 15 skipped.

**How to run:** Runtime → Run all. The first run in a fresh Colab runtime installs the pinned library versions and restarts the runtime once; then choose Run all again.

<details><summary>Source and conversion notes</summary>

Equivalent of scripts/reproduce_all.py followed by pytest.

Mechanical changes made when copying the .py files into this notebook:
`from src... import ...` lines are removed (the library code is defined in the cells above);
`sys.path.insert(...)` and `from __future__ import annotations` are removed;
`Path(__file__)...` repository roots are replaced by `REPO_ROOT` from the setup cell;
the command-line interface of `src/utils/hashing.py` is omitted;
command-line arguments of `reproduce_final_statistics.py` are replaced by the parameter cell;
`sys.exit(L.finish())` becomes `assert L.finish() == 0`.
</details>

In [ ]:
# ===== Setup: run this cell first =====
# REPO_SOURCE = "drive": the repository is in your Google Drive, either as the unzipped folder
#   `nids-partition-distortion-ae-dae` or as the zip file `nids-partition-distortion-ae-dae.zip`, anywhere
#   in My Drive (up to four folders deep). The cell finds it; set DRIVE_REPO only to point at one copy.
# REPO_SOURCE = "github": after the repository is on GitHub (fill in GITHUB_URL).
REPO_SOURCE = "drive"                                                    # "drive" or "github"
DRIVE_REPO = "/content/drive/MyDrive/nids-partition-distortion-ae-dae"   # optional hint
GITHUB_URL = "https://github.com/<your-account>/nids-partition-distortion-ae-dae.git"

import os, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "nids-partition-distortion-ae-dae"


def _is_repo(p: Path) -> bool:
    return (p / "artifacts" / "ARTIFACT_PROVENANCE.csv").is_file()


def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if _is_repo(p):
            return p
    raise FileNotFoundError(f"repository folder not found above {start}")


def _locate_in_drive(hint: Path, drive_root: Path) -> Path:
    # 1. the hint itself, a folder above it, or a nested copy below it (unzipping often creates name/name/)
    for c in [hint, *hint.parents, hint / REPO_NAME]:
        if _is_repo(c):
            return c
    # 2. the unzipped folder anywhere in My Drive, up to four levels deep
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + ["artifacts", "ARTIFACT_PROVENANCE.csv"])
        hits = sorted(drive_root.glob(pattern))
        if hits:
            if len(hits) > 1:
                print("Several copies found; using the first (set DRIVE_REPO to choose):", *[h.parents[1] for h in hits], sep="\n  ")
            return hits[0].parents[1]
    # 3. the zip file anywhere in My Drive (Google Drive does not unzip uploads): extract it to /content
    for depth in range(0, 4):
        zips = sorted(drive_root.glob("/".join(["*"] * depth + [REPO_NAME + "*.zip"])))
        if zips:
            dest = Path("/content/repo_from_zip")
            print(f"Found {zips[0]} - extracting to {dest} (results will be written there, not to Drive)")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            return _locate_in_drive(dest, dest)
    near = [p.name for p in drive_root.iterdir() if "nids" in p.name.lower() or "partition" in p.name.lower()] if drive_root.exists() else []
    raise FileNotFoundError(
        "Could not find the repository in Google Drive.\n"
        f"Looked for '{REPO_NAME}/artifacts/ARTIFACT_PROVENANCE.csv' (up to 4 folders deep) and for '{REPO_NAME}*.zip' in {drive_root}.\n"
        f"Items in My Drive whose names look related: {near or 'none'}.\n"
        "Fix: upload the zip file (or the unzipped folder) to My Drive, or set DRIVE_REPO to the folder that "
        "contains README.md, artifacts/ and notebooks/.")


_previous = globals().get("REPO_ROOT")                 # set by an earlier notebook in the same runtime (R00)
if _previous is not None and _is_repo(Path(_previous)):
    REPO_ROOT = Path(_previous)
elif IN_COLAB and REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = _locate_in_drive(Path(DRIVE_REPO), Path("/content/drive/MyDrive"))
elif IN_COLAB:
    target = Path("/content/" + REPO_NAME)
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(target)], check=True)
    REPO_ROOT = _find_repo(target)
else:                                     # local Jupyter: the notebook lives in notebooks/reproduction/
    REPO_ROOT = _find_repo(Path.cwd().resolve())
os.chdir(REPO_ROOT)

# The results were verified with these exact versions (the versions recorded by the Colab analysis).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
          "matplotlib": "3.10.0", "pytest": "8.3.5"}
wrong = {k: v for k, v in PINNED.items() if version(k) != v}
if wrong and IN_COLAB:
    print("Installing the pinned versions:", wrong)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    print("Done. The runtime will now restart; then run the notebook again from the top (Runtime > Run all).")
    os.kill(os.getpid(), 9)
elif wrong:
    print("WARNING: library versions differ from requirements.txt:", wrong)
print("Repository:", REPO_ROOT)
print("Python", sys.version.split()[0], "|", ", ".join(f"{k} {version(k)}" for k in PINNED))

In [ ]:
%run "notebooks/reproduction/R01_Model_View_Redundancy.ipynb"

In [ ]:
%run "notebooks/reproduction/R02_Partition_Structure.ipynb"

In [ ]:
%run "notebooks/reproduction/R03_Benign_Transfer_Analysis.ipynb"

In [ ]:
%run "notebooks/reproduction/R04_Selection_and_Final_Statistics.ipynb"

In [ ]:
%run "notebooks/reproduction/R05_Figures.ipynb"

In [ ]:
%run "notebooks/reproduction/R06_Claims_to_Artifacts_Map.ipynb"

In [ ]:
%run "notebooks/reproduction/R07_Tests.ipynb"

In [ ]:
import pandas as pd
for s in ["reproduce_fingerprints", "reproduce_partitions", "reproduce_transfer_analysis", "reproduce_final_statistics"]:
    log = pd.read_csv(REPO_ROOT / "results" / "claims_check" / f"{s}.csv")
    print(f"{s:32s} PASS {int((log.status == 'PASS').sum()):4d}   FAIL {int((log.status == 'FAIL').sum())}   notes {int((~log.status.isin(['PASS', 'FAIL'])).sum())}")
print("ALL CHECKS PASSED" if all((pd.read_csv(p).status != "FAIL").all() for p in (REPO_ROOT / "results" / "claims_check").glob("reproduce_*.csv")) else "SOME CHECKS FAILED")